# Top-5 POI XGBoost — OOF CV + Test Evaluation

Notebook trénuje a vyhodnocuje XGBoost model s 5 forward-selected POI příznaky
dle report.tex (tab. `tab:poi_correlation`).

**POI features:**
1. `transport_quality_score` — spojitá: 10/(1+d_metro) + 1/(1+d_tram)
2. `city_center_travel_time_min` — OSRM z CSV (train), Haversine fallback (test)
3. `convenience_nearest_km` — nearest convenience store
4. `grocery_avg_3_nearest_km` — avg 3 nearest groceries
5. `supermarket_nearest_km` — nearest supermarket

**Model:** TransformedTargetRegressor(log) + Model_pipeline(ColumnTransformer) + XGBRegressor
    tuned hyperparams: n=800, lr=0.025, depth=12, lossguide, max_leaves=63, subsample=0.7

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "NBS" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import joblib
import numpy as np
import pandas as pd
from sklearn.neighbors import BallTree
from sklearn.compose import TransformedTargetRegressor
from xgboost import XGBRegressor

from src.poi_models_workflow import (
    load_experiment_data, SELECTED_CLEANING_POLICY, CV,
    fit_cleaning_policy, prepare_xy, apply_cleaning_policy,
    out_of_fold_scores, fit_locked_model, score_predictions,
    make_config_with_extra_numeric, build_xgb_model,
)
from src.pipe import Model_pipeline

pd.options.display.float_format = "{:,.4f}".format
EARTH_RADIUS_KM = 6371.0088
RANDOM_STATE = 42

CITY_TRAVEL_PATH = PROJECT_ROOT / "data" / "city_center_travel_times.csv"
CITY_CENTERS_PATH = PROJECT_ROOT / "data" / "city_centers.csv"
MODELS_DIR = PROJECT_ROOT / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

TOP5_POI = [
    "transport_quality_score",
    "city_center_travel_time_min",
    "convenience_nearest_km",
    "grocery_avg_3_nearest_km",
    "supermarket_nearest_km",
]

print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"Travel times CSV exists: {CITY_TRAVEL_PATH.exists()}")

PROJECT_ROOT: /Users/honzanovak/bc/v2/v3/real-estate-prediction-ml/notebooks_data
Travel times CSV exists: True


## 1. Load Experiment Data

In [2]:
data = load_experiment_data()
train_fe = data.train_fe.copy()
test_fe = data.test_fe.copy()
transport = data.transport_poi
grocery = data.grocery_poi

print(f"Train FE: {train_fe.shape}")
print(f"Test FE:  {test_fe.shape}")
print(f"Transport POI: {transport.shape}")
print(f"Grocery POI:   {grocery.shape}")
print(f"POI kinds in transport: {sorted(transport['poi_kind'].dropna().unique())}")
print(f"POI kinds in grocery:   {sorted(grocery['poi_kind'].dropna().unique())}")

Train FE: (11921, 44)
Test FE:  (1263, 44)
Transport POI: (44272, 6)
Grocery POI:   (3286, 6)
POI kinds in transport: ['bus_stop', 'metro', 'public_transport', 'train_station', 'train_stop', 'tram_stop']
POI kinds in grocery:   ['convenience', 'supermarket']


## 2. Compute Top-5 POI Features

In [3]:
def build_transport_quality_score(
    apt_lat: np.ndarray, apt_lon: np.ndarray,
    transport_df: pd.DataFrame,
    w_metro: float = 10.0, w_tram: float = 1.0, w_bus: float = 0.0,
) -> np.ndarray:
    """
    Continuous transport quality score matching report.tex:
    Q = w_metro/(1+d_metro) + w_tram/(1+d_tram) + w_bus/(1+d_bus)
    """
    scores = np.zeros(len(apt_lat))
    apt_rad = np.radians(np.column_stack([apt_lat, apt_lon]))

    for kind, weight in [("metro", w_metro), ("tram_stop", w_tram), ("bus_stop", w_bus)]:
        if weight == 0:
            continue
        mask = transport_df["poi_kind"].str.lower() == kind
        if not mask.any():
            continue
        poi_rad = np.radians(
            transport_df.loc[mask, ["latitude", "longitude"]].astype(float).values
        )
        tree = BallTree(poi_rad, metric="haversine")
        dist_km, _ = tree.query(apt_rad, k=1)
        dist_km = dist_km.flatten() * EARTH_RADIUS_KM
        scores += weight / (1.0 + dist_km)

    return scores

In [4]:
def add_top5_poi_features(
    train_df: pd.DataFrame, test_df: pd.DataFrame,
    transport_df: pd.DataFrame, grocery_df: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Add 5 forward-selected POI features to train and test DataFrames.
    city_center_travel_time_min: OSRM from CSV for train, Haversine fallback for test.
    Returns modified copies.
    """
    train_df = train_df.copy()
    test_df = test_df.copy()

    apt_lat_tr = train_df["latitude"].astype(float).values
    apt_lon_tr = train_df["longitude"].astype(float).values
    apt_rad_tr = np.radians(np.column_stack([apt_lat_tr, apt_lon_tr]))

    apt_lat_te = test_df["latitude"].astype(float).values
    apt_lon_te = test_df["longitude"].astype(float).values
    apt_rad_te = np.radians(np.column_stack([apt_lat_te, apt_lon_te]))

    # ---- 1. transport_quality_score (continuous formula per report.tex) ----
    train_df["transport_quality_score"] = build_transport_quality_score(
        apt_lat_tr, apt_lon_tr, transport_df
    )
    test_df["transport_quality_score"] = build_transport_quality_score(
        apt_lat_te, apt_lon_te, transport_df
    )

    # ---- 2. supermarket_nearest_km ----
    sup_mask = grocery_df["poi_kind"].str.lower() == "supermarket"
    if sup_mask.any():
        sup_rad = np.radians(
            grocery_df.loc[sup_mask, ["latitude", "longitude"]].astype(float).values
        )
        sup_tree = BallTree(sup_rad, metric="haversine")
        train_df["supermarket_nearest_km"] = (
            sup_tree.query(apt_rad_tr, k=1)[0].flatten() * EARTH_RADIUS_KM
        )
        test_df["supermarket_nearest_km"] = (
            sup_tree.query(apt_rad_te, k=1)[0].flatten() * EARTH_RADIUS_KM
        )

    # ---- 3. convenience_nearest_km ----
    con_mask = ~sup_mask
    if con_mask.any():
        con_rad = np.radians(
            grocery_df.loc[con_mask, ["latitude", "longitude"]].astype(float).values
        )
        con_tree = BallTree(con_rad, metric="haversine")
        train_df["convenience_nearest_km"] = (
            con_tree.query(apt_rad_tr, k=1)[0].flatten() * EARTH_RADIUS_KM
        )
        test_df["convenience_nearest_km"] = (
            con_tree.query(apt_rad_te, k=1)[0].flatten() * EARTH_RADIUS_KM
        )

    # ---- 4. grocery_avg_3_nearest_km ----
    grocery_rad = np.radians(
        grocery_df[["latitude", "longitude"]].astype(float).values
    )
    grocery_tree = BallTree(grocery_rad, metric="haversine")
    dist_3_tr, _ = grocery_tree.query(apt_rad_tr, k=3)
    dist_3_te, _ = grocery_tree.query(apt_rad_te, k=3)
    train_df["grocery_avg_3_nearest_km"] = (
        np.mean(dist_3_tr, axis=1) * EARTH_RADIUS_KM
    )
    test_df["grocery_avg_3_nearest_km"] = (
        np.mean(dist_3_te, axis=1) * EARTH_RADIUS_KM
    )

    # ---- 5. city_center_travel_time_min (Haversine km) ----
    centers = pd.read_csv(CITY_CENTERS_PATH)
    clat = dict(zip(centers["locality_region_id"], centers["center_latitude"]))
    clon = dict(zip(centers["locality_region_id"], centers["center_longitude"]))

    for label, df in [("train", train_df), ("test", test_df)]:
        reg = df["locality_region_id"].astype(int).values
        c_lat = np.array([clat.get(r, np.nan) for r in reg])
        c_lon = np.array([clon.get(r, np.nan) for r in reg])
        lat_r = np.radians(df["latitude"].values)
        clat_r = np.radians(c_lat)
        dlat = clat_r - lat_r
        dlon = np.radians(c_lon - df["longitude"].values)
        a = np.sin(dlat/2)**2 + np.cos(lat_r)*np.cos(clat_r)*np.sin(dlon/2)**2
        df["city_center_travel_time_min"] = 2*EARTH_RADIUS_KM*np.arcsin(np.sqrt(np.clip(a,0,1)))
    print("city_center_travel_time_min (Haversine km) done")
    print("  Train mean:", round(train_df["city_center_travel_time_min"].mean(), 1), "km")
    print("  Test mean:", round(test_df["city_center_travel_time_min"].mean(), 1), "km")
    return train_df, test_df

In [5]:
train_fe, test_fe = add_top5_poi_features(train_fe, test_fe, transport, grocery)
print(f"\nAfter POI: train={train_fe.shape}, test={test_fe.shape}")
print(f"New columns: {[c for c in train_fe.columns if c in TOP5_POI]}")
print(f"\nNull check (train):")
print(train_fe[TOP5_POI].isnull().sum())
print(f"\nTransport quality stats: mean={train_fe['transport_quality_score'].mean():.3f}, "
      f"median={train_fe['transport_quality_score'].median():.3f}, "
      f"max={train_fe['transport_quality_score'].max():.3f}")

city_center_travel_time_min (Haversine km) done
  Train mean: 18.1 km
  Test mean: 17.9 km

After POI: train=(11921, 49), test=(1263, 49)
New columns: ['transport_quality_score', 'supermarket_nearest_km', 'convenience_nearest_km', 'grocery_avg_3_nearest_km', 'city_center_travel_time_min']

Null check (train):
transport_quality_score        0
city_center_travel_time_min    0
convenience_nearest_km         0
grocery_avg_3_nearest_km       0
supermarket_nearest_km         0
dtype: int64

Transport quality stats: mean=2.126, median=0.566, max=10.697


## 3. Build Tuned XGBoost Model

Hyperparams from report.tex Phase 2 GridSearchCV:
- n_estimators=800, lr=0.025, max_depth=12
- grow_policy=lossguide, max_leaves=63
- subsample=0.7, colsample_bytree=0.8, reg_lambda=1.0

Wrapper: TransformedTargetRegressor(log) + Model_pipeline(ColumnTransformer)
-> compatible with PredictLoader

In [6]:
def build_tuned_xgb_top5():
    """Build tuned top-5 POI XGBoost model -- PredictLoader-compatible."""
    config = make_config_with_extra_numeric(TOP5_POI)
    base = Model_pipeline(
        config=config,
        model_type="tree",
        model=XGBRegressor(
            n_estimators=800,
            learning_rate=0.025,
            max_depth=12,
            max_leaves=63,
            grow_policy="lossguide",
            subsample=0.7,
            colsample_bytree=0.8,
            reg_lambda=1.0,
            random_state=RANDOM_STATE,
            n_jobs=-1,
        ),
    )
    return TransformedTargetRegressor(
        regressor=base, func=np.log, inverse_func=np.exp
    )


def build_baseline_xgb():
    """Baseline XGBoost -- NO POI features, default thesis hyperparams."""
    return build_xgb_model()


print("Model builders ready.")

Model builders ready.


## 4. Out-of-Fold Cross-Validation (5-Fold)

In [7]:
print("Running OOF CV -- baseline XGBoost (no POI)...")
baseline_oof, _ = out_of_fold_scores(
    df=train_fe,
    model_builder=build_baseline_xgb,
    cleaning_policy=SELECTED_CLEANING_POLICY,
)
print("Baseline OOF:")
for k, v in baseline_oof.items():
    print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")

Running OOF CV -- baseline XGBoost (no POI)...


Baseline OOF:
  MedAPE: 0.0942
  MAPE: 0.1458
  wMAPE: 0.1314
  MAE: 1031201.1231
  R2: 0.8876
  n_scored: 11258


In [8]:
print("Running OOF CV -- tuned top-5 POI XGBoost...")
tuned_oof, _ = out_of_fold_scores(
    df=train_fe,
    model_builder=build_tuned_xgb_top5,
    cleaning_policy=SELECTED_CLEANING_POLICY,
)
print("Tuned Top-5 POI OOF:")
for k, v in tuned_oof.items():
    print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")

Running OOF CV -- tuned top-5 POI XGBoost...


Tuned Top-5 POI OOF:
  MedAPE: 0.0867
  MAPE: 0.1324
  wMAPE: 0.1228
  MAE: 963842.6630
  R2: 0.9000
  n_scored: 11258


In [9]:
comp = pd.DataFrame({
    "Baseline (no POI)": baseline_oof,
    "Top-5 POI tuned": tuned_oof,
}).T

print("\n=== OOF CV Comparison ===\n")
print(comp.to_string())
print(f"\nImprovement over baseline:")
for m in ["MedAPE", "MAPE", "wMAPE"]:
    pct = (baseline_oof[m] - tuned_oof[m]) / baseline_oof[m] * 100
    print(f"  {m}: {pct:+.2f}%")
print(f"  MAE:  {(baseline_oof['MAE'] - tuned_oof['MAE']):+,.0f} Kč")
print(f"  R²:   {(tuned_oof['R2'] - baseline_oof['R2']):+.4f} (abs)")


=== OOF CV Comparison ===

                   MedAPE   MAPE  wMAPE            MAE     R2    n_scored
Baseline (no POI)  0.0942 0.1458 0.1314 1,031,201.1231 0.8876 11,258.0000
Top-5 POI tuned    0.0867 0.1324 0.1228   963,842.6630 0.9000 11,258.0000

Improvement over baseline:
  MedAPE: +8.04%
  MAPE: +9.20%
  wMAPE: +6.53%
  MAE:  +67,358 Kč
  R²:   +0.0124 (abs)


## 5. Final Locked Model -- Test Set Evaluation

In [10]:
model, cleaned_test, preds, test_scores = fit_locked_model(
    train_df=train_fe,
    test_df=test_fe,
    model_builder=build_tuned_xgb_top5,
    cleaning_policy=SELECTED_CLEANING_POLICY,
)

print("=== Test Set Evaluation ===\n")
print(f"Model:  Top-5 POI XGBoost (tuned)")
print(f"MedAPE: {test_scores['MedAPE']:.4f}")
print(f"MAPE:   {test_scores['MAPE']:.4f}")
print(f"wMAPE:  {test_scores['wMAPE']:.4f}")
print(f"MAE:    {test_scores['MAE']:,.0f} Kč")
print(f"R²:     {test_scores['R2']:.4f}")

=== Test Set Evaluation ===

Model:  Top-5 POI XGBoost (tuned)
MedAPE: 0.0837
MAPE:   0.1239
wMAPE:  0.1148
MAE:    887,235 Kč
R²:     0.9176


## 6. Comparison with Report Claims

In [11]:
report_claims = {
    "MedAPE": 0.0825, "MAPE": 0.1265, "wMAPE": 0.1169,
    "MAE": 903_593, "R2": 0.9152,
}

_, _, baseline_preds, baseline_test = fit_locked_model(
    train_df=train_fe, test_df=test_fe,
    model_builder=build_baseline_xgb,
    cleaning_policy=SELECTED_CLEANING_POLICY,
)

compare = pd.DataFrame({
    "Baseline (test)": baseline_test,
    "Top-5 POI tuned (test)": test_scores,
    "Report claim": report_claims,
}).T

print(compare.to_string())
print(f"\nDelta vs report claim:")
for k in ["MedAPE", "MAPE", "wMAPE", "MAE", "R2"]:
    delta = test_scores[k] - report_claims[k]
    fmt = "{:+,.0f} Kč" if k == "MAE" else "{:+.4f}"
    print(f"  {k}: {fmt.format(delta)}")

                        MedAPE   MAPE  wMAPE          MAE     R2
Baseline (test)         0.0855 0.1327 0.1199 926,916.1875 0.9151
Top-5 POI tuned (test)  0.0837 0.1239 0.1148 887,235.0000 0.9176
Report claim            0.0825 0.1265 0.1169 903,593.0000 0.9152

Delta vs report claim:
  MedAPE: +0.0012
  MAPE: -0.0026
  wMAPE: -0.0021
  MAE: -16,358 Kč
  R2: +0.0024


In [12]:
# --- 6.5 Wilcoxon signed-rank test ---
from scipy.stats import wilcoxon

# Get y_test from cleaned_test (tuned model cell 15)
y_true = cleaned_test["price_total"].values

# Tuned predictions from cell 15 (variable `preds`)
# Baseline predictions from cell 17 (variable `baseline_preds`)
tuned_ape   = np.abs((y_true - preds) / y_true)
baseline_ape = np.abs((y_true - baseline_preds) / y_true)

stat, p_value = wilcoxon(baseline_ape, tuned_ape, alternative='greater')

print("Wilcoxon signed-rank test (one-sided, H₁: tuned < baseline):")
print(f"  Baseline median APE: {np.median(baseline_ape):.4f}")
print(f"  Tuned median APE:    {np.median(tuned_ape):.4f}")
print(f"  Test statistic: {stat:.1f}")
print(f"  p-value:        {p_value:.6f}")
if p_value < 0.05:
    print(f"  ✓ POI features SIGNIFICANTLY improve predictions at α = 0.05")
else:
    print(f"  ✗ No statistically significant improvement (p = {p_value:.4f})")


Wilcoxon signed-rank test (one-sided, H₁: tuned < baseline):
  Baseline median APE: 0.0855
  Tuned median APE:    0.0837
  Test statistic: 438563.0
  p-value:        0.001170
  ✓ POI features SIGNIFICANTLY improve predictions at α = 0.05


In [13]:
# Bootstrap 95% CI for MedAPE reduction
np.random.seed(42)
diffs = baseline_ape - tuned_ape  # positive = tuned is better
n_boot = 10_000
n = len(diffs)
boot_meds = np.array([np.median(np.random.choice(diffs, size=n, replace=True)) for _ in range(n_boot)])
ci_lo, ci_hi = np.percentile(boot_meds, [2.5, 97.5])
point_est = np.median(diffs)

print(f"Bootstrap 95% CI for MedAPE reduction (baseline − tuned):")
print(f"  Point estimate: {point_est:.4f}  ({point_est*100:.2f} pp absolute error reduction)")
print(f"  95% CI:         [{ci_lo:.4f}, {ci_hi:.4f}]")
print(f"  → Tuned model reduces median APE by {point_est*100:.2f} percentage points")
print(f"  → CI is {'entirely positive → improvement is robust' if ci_lo > 0 else 'includes zero → improvement may not be robust'}")


Bootstrap 95% CI for MedAPE reduction (baseline − tuned):
  Point estimate: 0.0024  (0.24 pp absolute error reduction)
  95% CI:         [0.0002, 0.0058]
  → Tuned model reduces median APE by 0.24 percentage points
  → CI is entirely positive → improvement is robust


## 7. Save Model

In [14]:
MODEL_PATH = MODELS_DIR / "tuned_xgb_top5_poi.joblib"
joblib.dump(model, MODEL_PATH)
print(f"Model saved to {MODEL_PATH}")
print(f"File size: {MODEL_PATH.stat().st_size / 1e6:.1f} MB")

Model saved to /Users/honzanovak/bc/v2/v3/real-estate-prediction-ml/notebooks_data/models/tuned_xgb_top5_poi.joblib
File size: 4.2 MB


In [15]:
print("Model type:", type(model).__name__)
r = model.regressor_
print("regressor_ type:", type(r).__name__)
print("Pipeline steps:", list(r.pipeline_.named_steps.keys()))
preproc = r.pipeline_.named_steps["preprocessing"]
feat_names = list(preproc.get_feature_names_out())
print(f"Total features after preprocessing: {len(feat_names)}")
poi_feat = [f for f in feat_names if any(p in f for p in TOP5_POI)]
print(f"POI features found: {len(poi_feat)}")
for pf in poi_feat:
    print(f"  {pf}")

Model type: TransformedTargetRegressor
regressor_ type: Model_pipeline
Pipeline steps: ['preprocessing', 'model']
Total features after preprocessing: 36
POI features found: 5
  num__transport_quality_score
  num__city_center_travel_time_min
  num__convenience_nearest_km
  num__grocery_avg_3_nearest_km
  num__supermarket_nearest_km


## Summary

Model saved to `models/tuned_xgb_top5_poi.joblib` — compatible with PredictLoader.